In [1]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
import findspark

findspark.init("/Users/i545672/spark3/spark-3.5.5-bin-hadoop3")
findspark.find()

'/Users/i545672/spark3/spark-3.5.5-bin-hadoop3'

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *

spark = (
    SparkSession
        .builder
        .appName("SparkTablesApp")
        .master("local[4]")
        .config("spark.dynamicAllocation.enabled", "false")
        .config("spark.sql.adaptive.enabled", "false")
        .enableHiveSupport()
        .getOrCreate()
)

sc= spark.sparkContext
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/24 22:32:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
yellowTaxiSchema = StructType([
 StructField("VendorID", IntegerType(), True),
 StructField("tpep_pickup_datetime", TimestampType(), True),
 StructField("tpep_dropoff_datetime", TimestampType(), True),
 StructField("passenger_count", DoubleType(), True),
 StructField("trip_distance", DoubleType(), True),
 StructField("RatecodeID", DoubleType(), True),
 StructField("store_and_fwd_flag", StringType(), True),
 StructField("PULocationID", IntegerType(), True),
 StructField("DOLocationID", IntegerType(), True),
 StructField("payment_type", IntegerType(), True),
 StructField("fare_amount", DoubleType(), True),
 StructField("extra", DoubleType(), True),
 StructField("mta_tax", DoubleType(), True),
 StructField("tip_amount", DoubleType(), True),
 StructField("tolls_amount", DoubleType(), True),
 StructField("improvement_surcharge", DoubleType(), True),
 StructField("total_amount", DoubleType(), True),
 StructField("congestion_surcharge", DoubleType(), True),
 StructField("airport_fee", DoubleType(), True),
])

yellowTaxisDF = spark.read.option("header", "true").schema(yellowTaxiSchema).csv(
    "./Files/YellowTaxis_202210.csv"
)
yellowTaxisDF.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [7]:
spark.sql("""SHOW DATABASES""").show()

+---------+
|namespace|
+---------+
|  default|
|  taxisdb|
+---------+



In [6]:
spark.sql("""CREATE DATABASE IF NOT EXISTS TaxisDB""")

25/05/24 22:33:15 WARN ObjectStore: Failed to get database taxisdb, returning NoSuchObjectException
25/05/24 22:33:15 WARN ObjectStore: Failed to get database taxisdb, returning NoSuchObjectException
25/05/24 22:33:15 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException
25/05/24 22:33:15 WARN ObjectStore: Failed to get database taxisdb, returning NoSuchObjectException


DataFrame[]

In [8]:
yellowTaxisDF.write.mode("overwrite").saveAsTable("TaxisDB.YellowTaxisManaged")

25/05/24 22:34:48 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
25/05/24 22:34:48 WARN HiveConf: HiveConf of name hive.internal.ss.authz.settings.applied.marker does not exist
25/05/24 22:34:48 WARN HiveConf: HiveConf of name hive.stats.jdbc.timeout does not exist
25/05/24 22:34:48 WARN HiveConf: HiveConf of name hive.stats.retries.wait does not exist


In [9]:
spark.sql("""SHOW TABLES IN TaxisDB""").show()

+---------+------------------+-----------+
|namespace|         tableName|isTemporary|
+---------+------------------+-----------+
|  taxisdb|yellowtaxismanaged|      false|
+---------+------------------+-----------+



In [10]:
spark.sql(""" SELECT * FROM TaxisDB.YellowTaxisManaged LIMIT 10""").show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2022-10-09 18:33:31|  2022-10-09 19:06:57|            1.0|         2.65|       1.0|                 N|         151|         142|           1|       21.0|  0.0|    0.5|      4.8

In [11]:
outputDF = spark.read.table("TaxisDB.YellowTaxisManaged")
outputDF.limit(10).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2022-10-09 18:33:31|  2022-10-09 19:06:57|            1.0|         2.65|       1.0|                 N|         151|         142|           1|       21.0|  0.0|    0.5|      4.8

In [14]:
spark.sql("""DESCRIBE TABLE EXTENDED TaxisDB.YellowTaxisManaged""").show(50, truncate=False)

+----------------------------+----------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                         |comment|
+----------------------------+----------------------------------------------------------------------------------+-------+
|VendorID                    |int                                                                               |NULL   |
|tpep_pickup_datetime        |timestamp                                                                         |NULL   |
|tpep_dropoff_datetime       |timestamp                                                                         |NULL   |
|passenger_count             |double                                                                            |NULL   |
|trip_distance               |double                                                                            |NULL   |
|RatecodeID             

In [15]:
yellowTaxisDF.write.mode("overwrite").option("path", "./Files/Output/YellowTaxisExtOutput.parquet").saveAsTable("TaxisDB.YellowTaxis")

In [16]:
spark.sql("""DESCRIBE TABLE EXTENDED TaxisDB.YellowTaxis""").show(50, truncate=False)

+----------------------------+---------------------------------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                                                |comment|
+----------------------------+---------------------------------------------------------------------------------------------------------+-------+
|VendorID                    |int                                                                                                      |NULL   |
|tpep_pickup_datetime        |timestamp                                                                                                |NULL   |
|tpep_dropoff_datetime       |timestamp                                                                                                |NULL   |
|passenger_count             |double                                                                                              

In [17]:
spark.sql(""" DROP TABLE IF EXISTS TaxisDB.YellowTaxis""")

DataFrame[]

In [ ]:
spark.sql(""" CREATE TABLE IF NOT EXISTS TaxisDB.YellowTaxis USING PARQUET LOCATION '/Users/i545672/SAPDevelop/Spark/spark-warehouse/taxisdb.db/Files/Output/YellowTaxisExtOutput.parquet/'""")

DataFrame[]

25/05/25 16:48:48 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 353727 ms exceeds timeout 120000 ms
25/05/25 16:48:48 WARN SparkContext: Killing executors is not supported by current scheduler.
25/05/25 16:48:56 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$